In [1]:
## Pull data from Google Books API 

import requests
from getpass import getpass

# Securely prompt for API key (input is hidden)
api_key = getpass("Enter your Google Books API key: ")

#Test API connection with a simple search query
url = "https://www.googleapis.com/books/v1/volumes"
params = {
    "q": "fiction",
    "maxResults": 40,
    "key": api_key  # Add API key to request
}

response = requests.get(url, params=params)
print(f"Status Code: {response.status_code}")
data = response.json()

# Check if request was successful
if response.status_code == 200 and 'items' in data:
    print(f"Number of items: {len(data['items'])}")
else:
    print(f"Error: {data.get('error', {}).get('message', 'Unknown error')}")

/Users/juliantaylor/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Status Code: 429
Error: Quota exceeded for quota metric 'Queries' and limit 'Queries per day' of service 'books.googleapis.com' for consumer 'project_number:624717413613'.


In [2]:
#Extract relevant information from each book item
def parse_book(item):
    volume = item.get("volumeInfo", {})
    
    return {
        "title": volume.get("title"),
        "authors": ", ".join(volume.get("authors", [])),
        "description": volume.get("description"),
        "categories": ", ".join(volume.get("categories", [])),
        "average_rating": volume.get("averageRating"),
        "ratings_count": volume.get("ratingsCount"),
        "published_year": volume.get("publishedDate", "")[:4]
    }

In [ ]:
#Collect data on multiple books from multiple genres
from tqdm import tqdm
import time

queries = [
    "fiction", "fantasy", "science fiction",
    "romance", "mystery", "history",
    "philosophy", "psychology"
]

all_books = []

for query in queries:
    for start in range(0, 200, 40):  # pagination
        params = {
            "q": query,
            "startIndex": start,
            "maxResults": 40,
            "key": api_key  # Include the API key from cell 1!
        }
        
        response = requests.get(url, params=params)
        data = response.json()
        
        items = data.get("items", [])
        print(f"Query '{query}' at index {start}: Found {len(items)} books")
        
        for item in items:
            book = parse_book(item)
            all_books.append(book)
        
        time.sleep(1)  # avoid rate limits

print(f"Total books collected: {len(all_books)}")

Query 'fiction' at index 0: Found 0 books
Query 'fiction' at index 40: Found 0 books
Query 'fiction' at index 80: Found 0 books
Query 'fiction' at index 120: Found 0 books
Query 'fiction' at index 160: Found 0 books
Query 'fantasy' at index 0: Found 0 books
Query 'fantasy' at index 40: Found 0 books
Query 'fantasy' at index 80: Found 0 books
Query 'fantasy' at index 120: Found 0 books
Query 'fantasy' at index 160: Found 0 books
Query 'science fiction' at index 0: Found 0 books
Query 'science fiction' at index 40: Found 0 books
Query 'science fiction' at index 80: Found 0 books
Query 'science fiction' at index 120: Found 0 books
Query 'science fiction' at index 160: Found 0 books
Query 'romance' at index 0: Found 0 books
Query 'romance' at index 40: Found 0 books
Query 'romance' at index 80: Found 0 books
Query 'romance' at index 120: Found 0 books
Query 'romance' at index 160: Found 0 books
Query 'mystery' at index 0: Found 0 books
Query 'mystery' at index 40: Found 0 books
Query 'myst

In [ ]:
#Convert data to dataframe  
import pandas as pd

df = pd.DataFrame(all_books)

# Clean up - remove duplicates first
df = df.drop_duplicates(subset=["title"])

# Only drop rows with missing descriptions if that column exists
if 'description' in df.columns:
    initial_rows = len(df)
    df = df.dropna(subset=["description"])
    removed = initial_rows - len(df)
    print(f"Removed {removed} rows without descriptions")

df.reset_index(drop=True, inplace=True)

print(f"Final shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Removed 387 rows without descriptions
Final shape: (377, 7)
Columns: ['title', 'authors', 'description', 'categories', 'average_rating', 'ratings_count', 'published_year']


,title,authors,description,categories,average_rating,ratings_count,published_year
0,Desire and Domestic Fiction,Nancy Armstrong,Desire and Domestic Fictionargues that far fro...,Language Arts & Disciplines,5.0,1.0,1987
1,Love and Sexuality in Dystopian Fiction. An An...,Lena Gräf,Seminar paper from the year 2015 in the subjec...,Literary Criticism,NaN,NaN,2016
2,Major Trends in the Post-independence Indian E...,"B. R. Agrawal, M. P. Sinha",This Book Presents A Reasonably Comprehensive ...,Indic fiction (English),NaN,NaN,2003
3,Fan Fiction and Fan Communities in the Age of ...,"Karen Hellekson, Kristina Busse",Fans have been responding to literary works si...,Literary Criticism,NaN,NaN,2014
4,"Peoria Public Library List of English Fiction,...","Peoria Public Library (Peoria, Ill.)",A catalog of juvenile and fiction books held b...,Children's literature,NaN,NaN,1894


In [ ]:
#Save dataset
df.to_csv("books_dataset.csv", index=False)

In [ ]:
## Load dataset

import pandas as pd

df = pd.read_csv("books_dataset.csv")

df = df[['title', 'authors', 'description', 'categories', 'average_rating','published_year']]
df = df.reset_index(drop=True)

df.head()
